# Notebook 3: Factor Risk Modeling & Convex Portfolio Optimization

Trong notebook này, chúng ta thực hiện chuyển đổi tín hiệu dự đoán AI ($\mu$) thành danh mục đầu tư theo tiêu chuẩn Quỹ phòng hộ quant:
1. Ước lượng ma trận hiệp phương sai rủi ro $\Sigma$ bằng kỹ thuật **Ledoit-Wolf Shrinkage**.
2. Tính toán hệ số rủi ro thị trường $eta$ của từng cổ phiếu so với **VN-Index**.
3. Giải bài toán tối ưu hóa Markowitz Mean-Variance bằng **CVXPY** với ràng buộc:
   - Gross Exposure = 100%
   - Market Neutral ($\sum w_i = 0$)
   - Beta Orthogonalization ($w^T eta = 0$).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.ai_models import AlphaMLP, AlphaPredictor
from src.risk_models import RiskModel, BetaCalculator
from src.optimization import PortfolioOptimizer

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Chuẩn bị dữ liệu và mô hình AI cho một ngày giao dịch mẫu


In [ ]:
fetcher = DataFetcher(use_mock_fallback=True)
raw_data = fetcher.fetch_all()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
predictor = AlphaPredictor(model)

target_date = '2024-01-15'
mu = predictor.predict_for_date(normalized_data['VCB'].set_index('date').reset_index(), target_date)
# Get all symbols predictions for target date
syms = [s for s in cleaned_data.keys() if s != Config.BENCHMARK_TICKER]
mu_dict = {}
for s in syms:
    sub = normalized_data[s][normalized_data[s]['date'] == target_date]
    if not sub.empty:
        mu_dict[s] = predictor.predict_for_date(normalized_data, target_date).get(s, np.random.uniform(-0.01, 0.01))
mu_series = pd.Series(mu_dict)


## 2. Ledoit-Wolf Covariance Matrix ($\Sigma$) & Market Beta ($eta$)


In [ ]:
risk_model = RiskModel()
cov_mat = risk_model.compute_covariance(cleaned_data, target_date)

beta_calc = BetaCalculator()
betas = beta_calc.compute_betas(cleaned_data, target_date)

print('Ma trận hiệp phương sai Ledoit-Wolf (10x10 đầu tiên):')
display(cov_mat.iloc[:10, :10])
print('Top 10 cổ phiếu có Beta nhạy cảm với VNINDEX lớn nhất:')
display(betas.sort_values(ascending=False).head(10))


## 3. Giải bài toán Markowitz Convex Optimization bằng CVXPY

$$\max_w \left( w^T \mu - rac{\lambda}{2} w^T \Sigma w 
ight) \quad 	ext{s.t.} \quad \sum |w_i| \le 1, \; \sum w_i = 0, \; w^T eta = 0$$


In [ ]:
optimizer = PortfolioOptimizer(risk_aversion=5.0, max_weight=0.10)
optimal_weights = optimizer.optimize(mu_series, cov_mat, betas)

print(f'Tổng trọng số Long: {optimal_weights[optimal_weights > 0].sum():.4f}')
print(f'Tổng trọng số Short: {optimal_weights[optimal_weights < 0].sum():.4f}')
print(f'Net Exposure (Long + Short): {optimal_weights.sum():.6f} (Target = 0.0)')
print(f'Systematic Portfolio Beta: {(optimal_weights * betas).sum():.6f} (Target = 0.0)')


In [ ]:
# Trực quan hóa phân bổ tỷ trọng danh mục tối ưu
plt.figure(figsize=(14, 5))
colors = ['green' if w > 0 else 'red' for w in optimal_weights]
optimal_weights.sort_values().plot(kind='bar', color=colors)
plt.title(f'Tỷ trọng Danh mục Tối ưu Trung lập Thị trường ngày {target_date}', fontweight='bold')
plt.ylabel('Tỷ trọng vị thế (w*)')
plt.xlabel('Mã cổ phiếu VN100')
plt.axhline(0, color='black', linewidth=1)
plt.tight_layout()
plt.show()
